[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/32_nlp/04_semantic_similarity_and_retrieval.ipynb)

# Bài 04 — Độ tương đồng ngữ nghĩa và tìm kiếm (retrieval)

**🎯 Mục tiêu**
- Các thước đo tương đồng văn bản: Jaccard, cosine trên TF-IDF, cosine trên embedding.
- Bài toán **retrieval** và cách đánh giá: Recall@k, MRR.
- Tìm kiếm FAQ tiếng Việt với truy vấn thật (có/không dấu): TF-IDF, n-gram ký tự, **BM25** from scratch; tác dụng của **bỏ dấu**.
- Tìm kiếm **ngữ nghĩa** (dense) vs **từ vựng** (lexical) khi truy vấn dùng từ đồng nghĩa; **hybrid** có tinh chỉnh trọng số.
- Bức tranh thực tế: sentence embedding, chỉ mục ANN, vector database, RAG.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/32_nlp

In [ ]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from ml_course.datasets import NEWS_LEXICON, NEWS_SYNONYMS, make_news_vi


def strip_accents(s):
    s = unicodedata.normalize("NFD", s.lower()).replace("đ", "d")
    return "".join(c for c in s if unicodedata.category(c) != "Mn")


LEXICON = {tuple(w.lower().split()) for w in NEWS_LEXICON}
MAX_LEN = max(map(len, LEXICON))


def segment(text):                                                     # tách từ khớp cực đại (bài 01)
    syl = re.findall(r"\w[\w\-]*", unicodedata.normalize("NFC", text).lower())
    tokens, i = [], 0
    while i < len(syl):
        for L in range(min(MAX_LEN, len(syl) - i), 0, -1):
            if L == 1 or tuple(syl[i:i + L]) in LEXICON:
                tokens.append("_".join(syl[i:i + L]))
                i += L
                break
    return tokens

## 1. Đo độ tương đồng giữa hai văn bản

- **Jaccard** trên tập từ: $|A\cap B| / |A\cup B|$ — đơn giản, không phân biệt từ quan trọng/không quan trọng.
- **Cosine trên TF-IDF**: tính trọng số theo độ hiếm (bài 02); vẫn chỉ khớp **đúng chữ**.
- **Cosine trên embedding** (trung bình vector từ, bài 03): từ đồng nghĩa gần nhau → khớp được **nghĩa**.

Embedding ở bài này dùng PPMI + SVD (bài 03) trên 20 000 tin không nhãn — không cần GPU.

In [ ]:
unlabeled = make_news_vi(20000, seed=1)
docs = [segment(t) for t in unlabeled.text]
counts = Counter(w for d in docs for w in d)
vocab = sorted(w for w, c in counts.items() if c >= 3)
w2i = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))
for d in docs:
    ids = [w2i[w] for w in d if w in w2i]
    for i, c in enumerate(ids):
        for j in range(max(0, i - 3), min(len(ids), i + 4)):
            if j != i:
                C[c, ids[j]] += 1
with np.errstate(divide="ignore"):
    ppmi = np.maximum(np.log(C * C.sum() / C.sum(1, keepdims=True) / C.sum(0, keepdims=True)), 0)
U, S, _ = np.linalg.svd(ppmi)
E = U[:, :50] * np.sqrt(S[:50])
E /= np.linalg.norm(E, axis=1, keepdims=True)


def embed(tokens):
    ids = [w2i[w] for w in tokens if w in w2i]
    v = E[ids].mean(0) if ids else np.zeros(E.shape[1])
    return v / (np.linalg.norm(v) + 1e-12)


tfidf_news = TfidfVectorizer(token_pattern=r"\S+").fit(" ".join(d) for d in docs)


def similarities(a, b):
    ta, tb = segment(a), segment(b)
    jac = len(set(ta) & set(tb)) / len(set(ta) | set(tb))
    va, vb = tfidf_news.transform([" ".join(ta), " ".join(tb)])
    return jac, (va @ vb.T).toarray()[0, 0], embed(ta) @ embed(tb)


examples = [("Samsung ra mắt điện thoại thông minh", "Samsung trình làng smartphone", "cùng nghĩa, khác chữ"),
            ("Samsung ra mắt điện thoại thông minh", "Apple nâng cấp máy tính bảng", "cùng chủ đề, khác nội dung"),
            ("Samsung ra mắt điện thoại thông minh", "Bệnh viện điều trị sốt xuất huyết", "khác chủ đề")]
for a, b, note in examples:
    jac, cos_tfidf, cos_emb = similarities(a, b)
    print(f"{note:26s}: Jaccard = {jac:.2f} | cosine TF-IDF = {cos_tfidf:.2f} | cosine embedding = {cos_emb:.2f}")

Câu "cùng nghĩa, khác chữ" gần như vô hình với Jaccard/TF-IDF (chỉ chung chữ "Samsung") nhưng có cosine embedding rất cao. Ngược lại, embedding trung bình cho câu **cùng chủ đề nhưng khác nội dung** điểm **cao y như vậy** — trung bình vector từ nắm "chủ đề" chứ không nắm "chi tiết" (ai làm gì với cái gì). Mỗi thước đo có điểm mù riêng.

## 2. Retrieval và cách đánh giá

Cho **truy vấn** $q$ và **kho tài liệu** $\{d_1, \dots, d_N\}$: chấm điểm mọi tài liệu, trả về danh sách xếp hạng. Với tập truy vấn có đáp án đúng (tài liệu liên quan):
- **Recall@k**: tỉ lệ truy vấn có tài liệu đúng nằm trong top-$k$.
- **MRR** (Mean Reciprocal Rank): trung bình $1/\text{hạng của tài liệu đúng đầu tiên}$ — thưởng xếp đúng ở vị trí cao.
- (Khi có nhiều tài liệu liên quan với mức độ khác nhau: nDCG — chương 26.)

In [ ]:
def evaluate(scores, gold, mask=None):
    """scores: (n_queries, n_docs); gold: chỉ số tài liệu đúng của mỗi truy vấn."""
    ranks = np.array([(s > s[g]).sum() + 1 for s, g in zip(scores, gold)])
    if mask is not None:
        ranks = ranks[mask]
    return {"R@1": np.mean(ranks == 1), "R@3": np.mean(ranks <= 3), "R@10": np.mean(ranks <= 10), "MRR": np.mean(1 / ranks)}

## 3. Tìm kiếm FAQ: truy vấn thật của người dùng

25 câu hỏi thường gặp của một shop online (câu hỏi + trả lời) và 75 truy vấn **viết tay** mô phỏng người dùng: diễn đạt khác, văn nói, **1/3 gõ không dấu**.

In [ ]:
faq = pd.read_csv("../../data/sample/faq_vi.csv")
queries = pd.read_csv("../../data/sample/faq_queries_vi.csv")
faq_docs = (faq.question + " " + faq.answer).tolist()
gold = (queries.faq_id - 1).values
no_accent = (queries["query"].map(strip_accents) == queries["query"].str.lower()).values
print(faq.head(3).to_string(index=False))
print(f"\n{len(queries)} truy vấn, {no_accent.sum()} không dấu. Ví dụ:", queries.sample(5, random_state=0)["query"].tolist())

### 3.1. BM25

BM25 (Robertson et al.) là hàm xếp hạng mặc định của các máy tìm kiếm (Elasticsearch, Lucene). So với TF-IDF: tần suất từ **bão hoà** (xuất hiện 10 lần không quan trọng gấp 10 lần 1 lần — tham số $k_1$) và chuẩn hoá theo **độ dài tài liệu** (tham số $b$):

$$\text{BM25}(q, d) = \sum_{t\in q}\text{idf}(t)\cdot\frac{f(t, d)\,(k_1 + 1)}{f(t, d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}, \qquad \text{idf}(t) = \ln\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

In [ ]:
class BM25:
    def __init__(self, tokenized_docs, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(d) for d in tokenized_docs]
        self.len = np.array([len(d) for d in tokenized_docs])
        self.avgdl = self.len.mean()
        N = len(tokenized_docs)
        df = Counter(w for d in tokenized_docs for w in set(d))
        self.idf = {w: np.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}

    def score(self, query_tokens):
        s = np.zeros(len(self.tf))
        for w in set(query_tokens):
            if w in self.idf:
                f = np.array([tf.get(w, 0) for tf in self.tf])
                s += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len / self.avgdl))
        return s


def tfidf_scores(prep, **kw):
    vec = TfidfVectorizer(sublinear_tf=True, **kw).fit([prep(d) for d in faq_docs])
    return (vec.transform([prep(q) for q in queries["query"]]) @ vec.transform([prep(d) for d in faq_docs]).T).toarray()


lower = str.lower
words = lambda s: re.findall(r"\w+", s)
bm25_faq = BM25([words(strip_accents(d)) for d in faq_docs])
methods = {
    "TF-IDF từ": tfidf_scores(lower),
    "TF-IDF từ + bỏ dấu": tfidf_scores(strip_accents),
    "TF-IDF ký tự 2–4": tfidf_scores(lower, analyzer="char_wb", ngram_range=(2, 4)),
    "TF-IDF ký tự 2–4 + bỏ dấu": tfidf_scores(strip_accents, analyzer="char_wb", ngram_range=(2, 4)),
    "BM25 + bỏ dấu": np.array([bm25_faq.score(words(strip_accents(q))) for q in queries["query"]]),
}
rows = []
for name, S in methods.items():
    r = evaluate(S, gold)
    rows.append({"phương pháp": name, **r, "R@1 (có dấu)": evaluate(S, gold, ~no_accent)["R@1"],
                 "R@1 (không dấu)": evaluate(S, gold, no_accent)["R@1"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

- **Bỏ dấu cả hai phía** là cải thiện lớn nhất: truy vấn không dấu từ khoảng một nửa lên gần như luôn đúng — một bước chuẩn hoá hai dòng code thắng mọi thay đổi mô hình. (Đánh đổi: trùng nghĩa sau khi bỏ dấu — bài 01; ở đây lợi ích lớn hơn nhiều.)
- N-gram ký tự không tốt hơn từ ở bộ này; BM25 xấp xỉ TF-IDF (tài liệu ngắn, độ dài tương tự nhau nên chuẩn hoá độ dài ít tác dụng).
- Truy vấn **có dấu** chỉ đúng top-1 khoảng 60%: phần còn lại là **lệch từ vựng** — người dùng diễn đạt khác hẳn câu hỏi trong FAQ:

In [ ]:
S = methods["TF-IDF từ + bỏ dấu"]
ranks = np.array([(s > s[g]).sum() + 1 for s, g in zip(S, gold)])
for i in np.where((ranks > 1) & ~no_accent)[0][:6]:
    print(f"truy vấn: {queries['query'][i]!r}\n   đúng: {faq.question[gold[i]]!r} (hạng {ranks[i]})\n   top-1: {faq.question[S[i].argmax()]!r}")

"trả lại món đồ" ↔ "đổi trả", "ship ... phí" ↔ "phí vận chuyển", "bỏ đơn" ↔ "hủy đơn", "trả tiền bằng cách nào" ↔ "hình thức thanh toán": từ chung (nếu có) lại là từ phổ biến ("mua", "tiền", "bao giờ") kéo về câu hỏi sai. Muốn khớp được, mô hình phải biết các cụm này **cùng nghĩa** — việc của biểu diễn ngữ nghĩa (embedding). Với FAQ thật, cách làm thực tế là sentence embedding tiền huấn luyện (mục 5); 25 tài liệu FAQ quá ít để tự học embedding. Mục 4 minh hoạ hiện tượng trên ngữ liệu tin tức, nơi có đủ văn bản không nhãn.

## 4. Lexical vs dense vs hybrid khi truy vấn dùng từ đồng nghĩa

Kho: 2 000 tin mới. Truy vấn: diễn đạt lại 300 tin trong kho bằng cách thay **mọi** từ có đồng nghĩa bằng một từ đồng nghĩa khác (`NEWS_SYNONYMS`) — tìm lại đúng tin gốc. Chia 100 truy vấn validation (để chọn trọng số hybrid) và 200 truy vấn test.

In [ ]:
syn_map = {}
for group in NEWS_SYNONYMS:
    for x in group:
        syn_map["_".join(x.lower().split())] = ["_".join(o.lower().split()) for o in group if o != x]

articles = make_news_vi(2000, seed=21)
art_docs = [segment(t) for t in articles.text]
rng = np.random.default_rng(3)
q_idx = rng.choice(len(art_docs), 300, replace=False)
q_docs = [[rng.choice(syn_map[t]) if t in syn_map else t for t in art_docs[i]] for i in q_idx]
changed = np.mean([np.mean([a != b for a, b in zip(art_docs[i], q)]) for i, q in zip(q_idx, q_docs)])
print(f"tỉ lệ token bị thay: {changed:.0%}")
print("gốc     :", " ".join(art_docs[q_idx[0]]))
print("truy vấn:", " ".join(q_docs[0]))

tf_art = TfidfVectorizer(token_pattern=r"\S+").fit(" ".join(d) for d in art_docs)
D_art = tf_art.transform(" ".join(d) for d in art_docs)
scores = {"TF-IDF": (tf_art.transform(" ".join(q) for q in q_docs) @ D_art.T).toarray()}
bm25_art = BM25(art_docs)
scores["BM25"] = np.array([bm25_art.score(q) for q in q_docs])
A_emb = np.array([embed(d) for d in art_docs])
scores["Dense (embedding TB)"] = np.array([embed(q) for q in q_docs]) @ A_emb.T

**Hybrid**: kết hợp điểm lexical và dense. Hai loại điểm có thang khác nhau → chuẩn hoá (z-score theo từng truy vấn) rồi trộn với trọng số $\alpha$, **chọn $\alpha$ trên tập validation** (không phải trên test):

$$s_{\text{hybrid}} = \alpha\cdot z(s_{\text{dense}}) + (1 - \alpha)\cdot z(s_{\text{BM25}})$$

In [ ]:
def zscore(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-12)


val, tst = np.arange(100), np.arange(100, 300)
gold_art = q_idx
alphas = np.linspace(0, 1, 11)
val_mrr = [evaluate(a * zscore(scores["Dense (embedding TB)"][val]) + (1 - a) * zscore(scores["BM25"][val]), gold_art[val])["MRR"] for a in alphas]
best_alpha = alphas[int(np.argmax(val_mrr))]
print("MRR validation theo α:", " | ".join(f"{a:.1f}: {m:.3f}" for a, m in zip(alphas, val_mrr)))
scores[f"Hybrid (α = {best_alpha:.1f})"] = best_alpha * zscore(scores["Dense (embedding TB)"]) + (1 - best_alpha) * zscore(scores["BM25"])
scores["Hybrid (α = 0.5, không tinh chỉnh)"] = 0.5 * zscore(scores["Dense (embedding TB)"]) + 0.5 * zscore(scores["BM25"])

print("\nkết quả trên 200 truy vấn test:")
print(pd.DataFrame([{"phương pháp": k, **evaluate(S[tst], gold_art[tst])} for k, S in scores.items()]).round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
test_table = pd.DataFrame([{"phương pháp": k, **evaluate(S[tst], gold_art[tst])} for k, S in scores.items()]).set_index("phương pháp")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 3]})
axes[0].plot(alphas, val_mrr, "o-")
axes[0].set_xlabel("α (trọng số của dense)")
axes[0].set_ylabel("MRR validation")
axes[0].set_title("Chọn α trên validation")
axes[1].barh(test_table.index[::-1], test_table["MRR"][::-1], color="C0")
axes[1].set_xlabel("MRR trên 200 truy vấn test")
axes[1].set_title("Lexical vs dense vs hybrid")
plt.tight_layout()
plt.show()

- Khi truy vấn và tài liệu dùng **từ khác nhau cho cùng khái niệm**, tìm kiếm dense vượt rõ lexical: TF-IDF/BM25 chỉ còn dựa vào phần chữ không đổi.
- Hybrid trộn đều (α = 0.5) **kém hơn** dense đơn thuần — điểm lexical ở đây chủ yếu là nhiễu; MRR validation tăng đều theo α và α tốt nhất là 1, tức bỏ hẳn lexical. Trong hệ thống thật, hybrid thường thắng vì truy vấn **có** chứa tín hiệu chỉ khớp đúng chữ mới bắt được (mã sản phẩm, tên riêng, số hiệu, thuật ngữ hiếm) mà embedding làm mờ đi. Trọng số phải **đo trên dữ liệu của mình**, không đặt theo cảm tính.
- Dense ở đây (trung bình vector từ) còn thô: không phân biệt được hai tin cùng chủ đề con khác chi tiết. Sentence embedding từ Transformer làm tốt hơn nhiều.

## 5. Trong thực tế: sentence embedding, ANN, RAG

- **Sentence embedding**: mô hình Transformer (bi-encoder) huấn luyện để câu cùng nghĩa có vector gần nhau — vd `sentence-transformers` với mô hình đa ngôn ngữ có tiếng Việt. Mã hoá kho tài liệu **một lần**, truy vấn mỗi lần một vector.
- **Re-ranking**: lấy top-50 bằng BM25/dense (nhanh), rồi xếp lại bằng cross-encoder (đọc cặp truy vấn–tài liệu cùng lúc, chính xác hơn nhưng chậm).
- **Tìm kiếm láng giềng gần đúng (ANN)**: với hàng triệu vector, không so với từng cái — chỉ mục HNSW, IVF/PQ (FAISS, các vector database) đổi một chút recall lấy tốc độ hàng nghìn lần.
- **RAG** (Retrieval-Augmented Generation): truy xuất các đoạn liên quan rồi đưa vào prompt cho LLM trả lời — chất lượng câu trả lời bị chặn trên bởi chất lượng retrieval; đánh giá retrieval (Recall@k) riêng trước khi đánh giá câu trả lời.

Đoạn code tham khảo (cần `pip install sentence-transformers` và tải mô hình ~500 MB — mặc định không chạy):

In [ ]:
RUN_ST = False                                                         # đổi thành True nếu đã cài sentence-transformers và có mạng

if RUN_ST:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    doc_emb = model.encode(faq_docs, normalize_embeddings=True)
    q_emb = model.encode(queries["query"].tolist(), normalize_embeddings=True)
    dense = q_emb @ doc_emb.T
    print("sentence embedding:", evaluate(dense, gold))
    hybrid = 0.5 * zscore(dense) + 0.5 * zscore(methods["BM25 + bỏ dấu"])
    print("hybrid với BM25   :", evaluate(hybrid, gold))
else:
    print("Bỏ qua phần sentence-transformers (RUN_ST = False).")

| Tình huống | Gợi ý |
|---|---|
| Truy vấn chứa mã, tên riêng, thuật ngữ chính xác | Lexical (BM25) — không bỏ qua |
| Người dùng diễn đạt tự do, đồng nghĩa, nhiều cách nói | Dense (sentence embedding) |
| Hệ thống thật, đa dạng truy vấn | Hybrid + re-ranking, trọng số chọn trên tập đánh giá |
| Người dùng gõ không dấu / sai chính tả | Chuẩn hoá (bỏ dấu), n-gram ký tự, sửa lỗi chính tả |
| Kho rất lớn | Chỉ mục ANN (HNSW/IVF) cho dense; chỉ mục ngược (inverted index) cho lexical |

## 🧠 Kiểm tra nhanh

1. Cosine trên TF-IDF và cosine trên embedding khác nhau thế nào? Mỗi cái có điểm mù gì?
2. Định nghĩa Recall@k và MRR.
3. BM25 khác TF-IDF ở những điểm nào?
4. Vì sao bỏ dấu cải thiện mạnh tìm kiếm FAQ?
5. Khi nào hybrid tốt hơn dense đơn thuần? Vì sao phải chọn trọng số trên tập validation?
6. RAG là gì? Vì sao cần đánh giá retrieval riêng?

## 📌 Tóm tắt

- Tương đồng lexical (Jaccard, TF-IDF, BM25) khớp chữ; dense (embedding) khớp nghĩa nhưng có thể làm mờ chi tiết.
- Đánh giá retrieval bằng Recall@k, MRR trên tập truy vấn có đáp án; tách validation để tinh chỉnh.
- Chuẩn hoá văn bản (bỏ dấu) là cải thiện rẻ và lớn nhất cho người dùng tiếng Việt.
- Hệ thống thực tế: sentence embedding + BM25 (hybrid), re-ranking, ANN; RAG dựa trên retrieval tốt.

## 📚 Tài liệu tham khảo

- Robertson & Zaragoza (2009) — *The Probabilistic Relevance Framework: BM25 and Beyond*
- Reimers & Gurevych (2019) — *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*
- Karpukhin et al. (2020) — *Dense Passage Retrieval for Open-Domain Question Answering*
- Lewis et al. (2020) — *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*
- Malkov & Yashunin (2018) — *Efficient and robust approximate nearest neighbor search using HNSW*
- Manning, Raghavan & Schütze — *Introduction to Information Retrieval*, chương 6, 8, 11